# 01 - Ingestion et reduction (Telecom Milan)

> **📁 Résultat de cette phase :** Le dataset agrégé est sauvegardé dans `data/processed/work_600cells.parquet` et `data/processed/work_1024cells.parquet`.

Objectif de cette etape: construire un dataset de travail propre (`600 cellules`, pas `30 min`) et l'enregistrer en Parquet.

Choix assumes ici (coherents avec le pipeline):
- agreger tous les `country_code` pour avoir le trafic total reel,
- passer de 10 min a 30 min pour reduire le bruit et la volumetrie,
- garder 500 cellules actives + 100 calmes (stratification geographique en 4 quadrants).

In [1]:
from pathlib import Path
import polars as pl

ROOT = Path("..")
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUT_PATH = PROCESSED_DIR / "work_600cells.parquet"
RANDOM_SEED = 42

raw_files = sorted(RAW_DIR.glob("sms-call-internet-mi-*.txt"))
print(f"Fichiers detectes: {len(raw_files)}")
if not raw_files:
    raise FileNotFoundError("Aucun fichier brut trouve dans data/raw/")

Fichiers detectes: 62


In [2]:
# Ingestion + aggregation
# 1) On lit tous les fichiers texte sans header
# 2) On remplace les valeurs vides par 0 pour les compteurs utiles
# 3) On construit un slot 30 min
# 4) On somme internet/sms_in/call_in par cellule et par slot (tous country_code inclus)

lf = (
    pl.scan_csv(
        [str(p) for p in raw_files],
        separator="\t",
        has_header=False,
        schema_overrides={
            "column_1": pl.Int32,
            "column_2": pl.Int64,
            "column_3": pl.Int32,
            "column_4": pl.Float64,
            "column_5": pl.Float64,
            "column_6": pl.Float64,
            "column_7": pl.Float64,
            "column_8": pl.Float64,
        },
        null_values=[""],
        ignore_errors=True,
        rechunk=False,
    )
    .rename({
        "column_1": "square_id",
        "column_2": "time_interval",
        "column_3": "country_code",
        "column_4": "sms_in",
        "column_5": "sms_out",
        "column_6": "call_in",
        "column_7": "call_out",
        "column_8": "internet",
    })
    .with_columns([
        pl.col("sms_in").fill_null(0.0),
        pl.col("call_in").fill_null(0.0),
        pl.col("internet").fill_null(0.0),
        ((pl.col("time_interval") // 1000) // 1800 * 1800).alias("slot_30m"),
    ])
    .group_by(["square_id", "slot_30m"])
    .agg([
        pl.sum("internet").alias("internet_volume"),
        pl.sum("sms_in").alias("sms_in"),
        pl.sum("call_in").alias("call_in"),
    ])
)

aggregated = lf.collect(engine="streaming")
print("Lignes apres aggregation:", aggregated.height)

Lignes apres aggregation: 29753511


In [3]:
# Controle de completude (>95%) avant la selection finale des 600 cellules
t_min = aggregated.select(pl.min("slot_30m")).item()
t_max = aggregated.select(pl.max("slot_30m")).item()
expected_slots = int((t_max - t_min) // 1800 + 1)

completeness_all = (
    aggregated.group_by("square_id")
    .agg(pl.col("slot_30m").n_unique().alias("n_slots"))
    .with_columns((pl.col("n_slots") / expected_slots * 100).alias("completeness_pct"))
)

eligible_cells = completeness_all.filter(pl.col("completeness_pct") >= 95).select("square_id")
aggregated = aggregated.join(eligible_cells, on="square_id", how="inner")
print("Cellules eligibles (>=95%):", eligible_cells.height)

Cellules eligibles (>=95%): 9994


In [4]:
# Selection des 600 cellules
# - top 500 actives selon la moyenne internet
# - 100 calmes, tirees par quadrant (25 chacun)

activity = (
    aggregated.group_by("square_id")
    .agg(pl.mean("internet_volume").alias("mean_internet_volume"))
    .with_columns(
        pl.when(pl.col("square_id") <= 2500).then(1)
        .when(pl.col("square_id") <= 5000).then(2)
        .when(pl.col("square_id") <= 7500).then(3)
        .otherwise(4)
        .alias("quadrant")
    )
)

top_500 = activity.sort("mean_internet_volume", descending=True).head(500)
top_ids = set(top_500["square_id"].to_list())
quiet_pool = activity.filter(~pl.col("square_id").is_in(list(top_ids)))

quiet_parts = []
for q in [1, 2, 3, 4]:
    q_df = quiet_pool.filter(pl.col("quadrant") == q)
    quiet_parts.append(q_df.sample(n=min(25, q_df.height), seed=RANDOM_SEED + q, shuffle=True))

quiet_100 = pl.concat(quiet_parts)
if quiet_100.height < 100:
    missing = 100 - quiet_100.height
    already = set(quiet_100["square_id"].to_list()) | top_ids
    extra = quiet_pool.filter(~pl.col("square_id").is_in(list(already))).sample(
        n=min(missing, quiet_pool.height), seed=RANDOM_SEED + 99, shuffle=True
    )
    quiet_100 = pl.concat([quiet_100, extra])

selected_ids = top_500.select("square_id").vstack(quiet_100.select("square_id")).unique()
assert selected_ids.height == 600, f"Selection invalide: {selected_ids.height}"

work_df = aggregated.join(selected_ids, on="square_id", how="inner").sort(["square_id", "slot_30m"])
print("Lignes finales:", work_df.height)

Lignes finales: 1785594


In [5]:
# Sauvegarde parquet + controles qualite finaux
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
work_df.write_parquet(OUTPUT_PATH, compression="zstd")

duplicates = work_df.select(pl.len() - pl.struct(["square_id", "slot_30m"]).n_unique()).item()
n_cells = int(work_df.select(pl.col("square_id").n_unique()).item())
n_slots_total = int(work_df.select(pl.col("slot_30m").n_unique()).item())

completeness = (
    work_df.group_by("square_id")
    .agg(pl.col("slot_30m").n_unique().alias("n_slots"))
    .with_columns((pl.col("n_slots") / expected_slots * 100).alias("completeness_pct"))
)

min_completeness = float(completeness.select(pl.min("completeness_pct")).item())
pct_above_95 = float(completeness.select((pl.col("completeness_pct") >= 95).mean() * 100).item())

print("=== CONTROLES QUALITE ===")
print("n_cells:", n_cells)
print("n_slots_total:", n_slots_total)
print("duplicates:", int(duplicates))
print("slot_30m_min_epoch_s:", int(t_min))
print("slot_30m_max_epoch_s:", int(t_max))
print("expected_slots_per_cell:", expected_slots)
print("min_completeness_pct:", round(min_completeness, 2))
print("cells_above_95pct_completeness:", round(pct_above_95, 2))
print("Parquet ecrit:", OUTPUT_PATH)

=== CONTROLES QUALITE ===
n_cells: 600
n_slots_total: 2976
duplicates: 0
slot_30m_min_epoch_s: 1383260400
slot_30m_max_epoch_s: 1388615400
expected_slots_per_cell: 2976
min_completeness_pct: 99.8
cells_above_95pct_completeness: 100.0
Parquet ecrit: C:\Users\hp\OneDrive\Desktop\projectTimeSeries\data\processed\work_600cells.parquet
